In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st

In [ ]:
homer_path = ''
env = ''

In [ ]:
os.system(env+'python ./metilene3/metilene3.py \
    -i ./data/GSE186458_blood.input.tsv \
    -o ./Blood \
    -t 16 \
    -n 3 \
    -plot True\
')

In [ ]:
os.system('cp ./Blood/DMRs-unsupervised.tsv ../SourceData/Fig.4b.txt')

In [ ]:
renameG = {
    'G0':'A',
    'G1':'B',
    'G2':'C',
    'G3':'D',
    'G4':'E',
    'G5':'F',
}

blood_s = pd.read_table('./Blood/DMRs.tsv')
blood_s['hypomethylated'] = blood_s['Hypo-groups']
blood_s['intermediate'] = blood_s['Int-groups']
blood_s['hypermethylated'] = blood_s['Hyper-groups']
for i in renameG.keys():
    blood_s['hypomethylated'] = blood_s['hypomethylated'].str.replace(i,renameG[i])
    blood_s['intermediate'] = blood_s['intermediate'].str.replace(i,renameG[i])
    blood_s['hypermethylated'] = blood_s['hypermethylated'].str.replace(i,renameG[i])
blood_s['mode'] = 'supervised'

# blood = pd.concat([blood_u, blood_s]).sort_values(['mode','chr','start'])
blood = pd.concat([blood_s]).sort_values(['length','p-kwt'], ascending=[0,1])
blood = blood['chr	start	stop	meandiffabs	length	p-kwt	hypomethylated	intermediate	hypermethylated'.split('\t')]
blood.to_csv('./figures/ST2.tsv',sep='\t', index=False)
blood

In [ ]:
from Bio import Phylo
import matplotlib.pyplot as plt

tree = Phylo.read("./Blood/DMTree.nwk", "newick")

import seaborn as sns
colors = pd.read_table('./Blood/clusters.tsv', index_col=0)
colors['group'] = [sns.color_palette("Set2")[int(i.split('G')[1])] for i in colors['Group']]
colors['subtype'] = [i.split('=')[1].split('-')[1] for i in colors.index]
colors['subtype'] = colors['subtype'].map((colors[['subtype','group']].groupby('subtype').first()['group'].apply(lambda x:tuple([i*0.85 for i in x]))).to_dict())
colors.index = [i.replace('Z00000','') for i in colors.index]
colors.head()

def change_labels(clade):
    if clade.name:
        clade.name = clade.name.split('Blood-')[-1].replace('Z00000','')+'-'.join(['' for i in range(15)])
    for subclade in clade.clades:
        change_labels(subclade)

change_labels(tree.root)

cmap = colors['group'].to_dict()
for i in colors.index:
    cmap[i.split('Blood-')[-1].split('_')[-1].replace('Z00000','')+'-'.join(['' for i in range(15)])] = cmap[i]
f,a = plt.subplots(figsize=[15,10])
Phylo.draw(tree, axes=a, do_show=False, label_colors=cmap)
plt.xlim([-30,3100])
a.spines['top'].set_visible(False)
a.spines['left'].set_visible(False)
a.spines['right'].set_visible(False)
a.set_xlabel(None)
a.yaxis.set_visible(False)
plt.savefig('./figures/4b.pdf', bbox_inches='tight')

In [ ]:
dmrmean_m_rename = pd.read_table('./Blood/heatmap.tsv', index_col=0)
dmrmean_m_rename.index = [i.split(' ')[-1].replace('Z00000','') for i in dmrmean_m_rename.index]

colors = colors.loc[dmrmean_m_rename.index]
cm = sns.clustermap(dmrmean_m_rename,\
        row_colors=[colors['group'],\
                    (colors['subtype']==1).map({False:'white'}),\
                    colors['subtype'],\
                    (colors['subtype']==1).map({False:'white'}),\
                    ],\
        # row_linkage=lk.linkage,\
        col_cluster=False,row_cluster=False,\
        cmap='Spectral_r', dendrogram_ratio=0.000001, xticklabels=False, yticklabels=False, \
        method='ward', cbar_pos=None, vmax=1, vmin=0, center=0.5, colors_ratio=0.03)

plt.savefig('./figures/4b-r.pdf', bbox_inches='tight')

In [ ]:
met = pd.read_table('./data/GSE186458_blood.input.tsv', na_values=['.']).dropna()
met.columns = [i.split('=')[-1] for i in met.columns]
met

In [ ]:
cpgstd = met.drop(columns=['chrom','end']).T.std()
cpgstd

In [ ]:
dmrmean_m_rename = pd.read_table('./Blood/heatmap.tsv', index_col=0)
sinfo = dmrmean_m_rename[[]]
sinfo['group'] = [i.split(' ')[0] for i in dmrmean_m_rename.index]
sinfo['subtype'] = [i.split('=')[1].split('-')[1]+'-naive' if i.find('Naive')>-1 else i.split('=')[1].split('-')[1] for i in dmrmean_m_rename.index]
sinfo.index = [i.split('=')[-1] for i in sinfo.index]

typec = {}
for i in sinfo['group'].unique():
    typec[i] = sns.color_palette("Set2")[int(i.split('G')[1])]

sinfo

In [ ]:
udmrs = pd.read_table('./Blood/DMRs-unsupervised.tsv',skiprows=2)
dmtncpg = udmrs.loc[(udmrs['meandiffabs']>0.5)&(udmrs['#Hyper']>=2)&(udmrs['#Hypo']>=2)]['length'].sum()
dmtncpg

In [ ]:
udmr4pca = []
udmrs.loc[(udmrs['meandiffabs']>0.5)&(udmrs['#Hyper']>=2)&(udmrs['#Hypo']>=2)]['mean'].apply(lambda x:udmr4pca.append(x.split('|')))
udmr4pca = pd.DataFrame(udmr4pca).astype(float).T
udmr4pca.index = met.columns[2:]
udmr4pca

In [ ]:
f,ax = plt.subplots(1,4,figsize=[12,3])

import numpy as np
from sklearn.decomposition import PCA


sd_blood_pcas = []
models = ['all CpGs','top 1% CpGs','eq. #CpGs','unsupervised DMRs']

for ii, tmp in enumerate([
    np.array(met.drop(columns=['chrom','end']).T),
    np.array(met.drop(columns=['chrom','end']).loc[cpgstd>cpgstd.quantile(0.99)].T),
    np.array(met.drop(columns=['chrom','end']).loc[cpgstd>=list(cpgstd.sort_values())[-dmtncpg]].T),
    udmr4pca
]):
    pca = PCA(n_components=2)
    print(tmp.shape)
    X = pd.DataFrame(pca.fit_transform(tmp))
    X['model'] = models[ii]
    sd_blood_pcas.append(X)
    print(pca.explained_variance_ratio_)
    X.index = met.columns[2:]
    X['grp'] = X.index.map(sinfo['group'])
    X['subtype'] = X.index.map(sinfo['subtype'])
    a = sns.scatterplot(x=X[0],y=X[1],hue=X['grp'],s=50, palette=typec, ax=ax[ii], legend=None)
    a.spines['top'].set_visible(False)
    # a.spines['left'].set_visible(False)
    a.spines['right'].set_visible(False)
    # a.yaxis.set_visible(False)
    a.set_title('n='+str(tmp.shape[1]))
    a.set_ylabel('PC2('+str(pca.explained_variance_ratio_[1]*100)[:5]+'%)')
    a.set_xlabel('PC1('+str(pca.explained_variance_ratio_[0]*100)[:5]+'%)')
    a.set_box_aspect(1)
    a.set_xticks(np.linspace(*a.get_xlim(), 3))
    a.set_yticks(np.linspace(*a.get_ylim(), 3))


f.tight_layout()
plt.savefig('./figures/ED5b.pdf', bbox_inches='tight')

In [ ]:
pd.concat(sd_blood_pcas).to_csv('../SourceData/Fig.ED5b.txt',sep='\t')
pd.concat(sd_blood_pcas)

In [ ]:
rename_dict = {
    'P2_2_3_3_3_3': 'Mono+Gran(hypo) vs B+NK+T+naiveT',
    'N0_0_0_0_2_3': 'T vs naiveT(hypo)',
}

In [ ]:
dmrs = pd.read_table('./Blood/DMRs.tsv').sort_values(['chr','start','stop'])
    
dmrs['strand'] = '+'
dmrs['stop_1'] = dmrs['stop']+1

path = './Blood/motif/'
os.system('mkdir '+path)
for i in ['P2|2|3|3|3|3', 'N0|0|0|0|2|3']:
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv('./Blood/motif/'+i.replace('|','_')+'.bed', sep='\t', header=False)
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(~dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv('./Blood/motif/'+i.replace('|','_')+'.anti.bed', sep='\t', header=False)
    
    os.system(env+homer_path+'findMotifsGenome.pl ./Blood/motif/'+i.replace('|','_')+'.bed'+\
                  ' hg19'+\
                  ' ./Blood/motif/'+i.replace('|','_')+' -bits -size 250  -bg ./Blood/motif/'+i.replace('|','_')+'.anti.bed')

In [ ]:
os.system('cp ./Blood/motif/P2_2_3_3_3_3.bed ../SourceData/Fig.4c-part1.txt')
os.system('cp ./Blood/motif/P2_2_3_3_3_3.anti.bed ../SourceData/Fig.4c-part2.txt')
os.system('cp ./Blood/motif/N0_0_0_0_2_3.bed ../SourceData/Fig.4c-part3.txt')
os.system('cp ./Blood/motif/N0_0_0_0_2_3.anti.bed ../SourceData/Fig.4c-part4.txt')

In [ ]:
os.system('cp ./Blood/motif/P2_2_3_3_3_3/knownResults.html ./figures/4c.html')
os.system('cp ./Blood/motif/N0_0_0_0_2_3/knownResults.html ./figures/4c-b.html')